# Lesson 1 : Create a basic agent

In this exercise, we learn the fundamentals of the way to build and run agent in Agent Framework.

Agent Framework supports various backend clients - such as, OpenAI Assistants API, OpenAI Responses API, Anthropic Client, Ollama, Foundry agent, etc. For example, when you want to work with Anthropic Claude API in Agent Framework, you can use ```AnthropicClient``` in Agent Framework SDK.

Throughout this workshop, we use a client ```FoundryChatClient```, which is backed by Azure AI Projects SDK (```azure-ai-projects```) version 2 to connect to Microsoft Foundry (Foundry v2).

Before starting, please remember to perform the preparations described in [Readme.md](./Readme.md).

## Basic (Synchronous responses)

Firstly, we create a client object to connect to Microsoft Foundry as follows.  
All the difference depending on individual clients is handled on this client's object (i.e., ```FoundryChatClient```).

> Note : In Foundry agent, Azure OpenAI Responses API is used to invoke the request internally. (Therefore ```FoundryChatClient``` inherits ```RawOpenAIChatClient``` at the bottom.)

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

In this example, we'll use local tool calling in the agent, and we then define local functions as follows.

In [2]:
# define local tools
from agent_framework import tool
from typing import Annotated
from pydantic import Field
from random import randint

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

Create an agent using above client and define tool functions as follows.

As I have mentioned above, all the difference depending on individual clients is handled on client's object. The below ```Agent``` class is then a generic agent class, not depending on client.

> Note : ```Agent``` class inherits ```BaseAgent``` class, and some agents constitute its unique class by inheriting ```BaseAgent``` - e.g., ```ClaudeAgent``` which uses Claude Agent SDK at the bottom, ```GitHubCopilotAgent``` which uses GitHub Copilot SDK at the bottom, etc. You can also easily build your own custom agent for Agent Framework from scratch.  
> **```BaseAgent``` class is the center of Agent Framework.**

> Note : When you use the existing Foundry agent (generated in Foundry Portal), also use the dedicated ```FoundryAgent``` instead.

In [3]:
from agent_framework import Agent

# create agent
agent = Agent(
    name="BasicWeatherAgent",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[get_weather, get_temperature])

Let's run the agent through Microsoft Foundry.  
You'll see that the function tools are being used appropriately.

In [4]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka.")
display(Markdown(result.text))

Osaka is **cloudy**, and the temperature is **22 °C**.

## Trace internal messages

Let's see the list of internal messages.  
You'll see that ```get_weather()``` and ```get_temperature()``` are concurrently called by the function calling.

In [5]:
import agent_framework

for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if c.type == "function_call":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif c.type == "function_result":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif c.type == "text":
            print(f"*** {c.type} ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {c.type}***")

********** message 0 **********
*** function_call ***
call id : call_3JWiYmxuafK82rtIlMUhYd3J
function name : get_weather
function arguments : {"location":"Osaka"}
*** function_call ***
call id : call_UGV7t0Be9fWRsm2RFXwq2iyz
function name : get_temperature
function arguments : {"location":"Osaka"}
********** message 1 **********
*** function_result ***
call id : call_3JWiYmxuafK82rtIlMUhYd3J
function result : The weather in Osaka is cloudy.
exceptions : None
*** function_result ***
call id : call_UGV7t0Be9fWRsm2RFXwq2iyz
function result : The temperature in Osaka is 22 degrees.
exceptions : None
********** message 2 **********
*** text ***
text : Osaka is **cloudy**, and the temperature is **22 °C**.


## Asynchronous responses

For the fluent user experience, you can also perform streaming outputs as follows. (Each character will be displayed one by one.)

In [6]:
async def streaming_example():
    async for chunk in agent.run(
        "Tell me the weather and temperature in Osaka.",
        stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

await streaming_example()

Osaka weather: rainy.  
Osaka temperature: 14 °C.



## Background responses

By setting the following option, you can also run the foreground tasks, polling until the operation completes.

> Note : Background responses cannot be used together with local function tools, because local functions run in the foreground. (I have then set prompt not to use local function tools in the following example.)

In [7]:
import asyncio

result = await agent.run(
    "Write a summary of the weather in five bullet points when it's stormy.",
    options={"background": True},
)

while result.continuation_token is not None:
    await asyncio.sleep(1)
    print("running ...")
    result = await agent.run(
        options={"continuation_token": result.continuation_token},
    )

display(Markdown(result.text))

running ...
running ...


- Severe thunderstorms in the area, with frequent lightning and periods of torrential rain.  
- Strong, gusty winds that can bring down tree limbs and make travel difficult.  
- Rapidly changing conditions: visibility drops quickly in heavy downpours and squall lines.  
- Elevated risk of localized flooding in low-lying or poor-drainage spots.  
- Potential for hail and isolated tornadoes, depending on storm intensity and instability.